# Pool de escalabilidad (E5-base)

Vectoriza abstracts en tramos de 50 000, de unos 150 MB cada uno, hasta el número de tramos elegido en la
celda 3. **Comprueba antes que Drive tenga ese espacio libre**: si se llena, los archivos no se suben.

**Cómo usarlo**

1. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4 → Guardar**.
2. Menú **Entorno de ejecución → Ejecutar todo**, y autoriza el acceso a Google Drive cuando lo pida.
3. Deja la pestaña abierta, con la tapa de la laptop abierta y sin suspender el equipo.

**Si se corta o se acaba la GPU gratis:** vuelve a **Ejecutar todo**, aunque sea otro día. Lo que ya está
hecho se salta.

Cada línea de avance indica cuánto tardó ese tramo. Todo queda en `tabd_scale_e5_base/` en Google Drive.

In [ ]:
#@title 1. Preparar el entorno
import base64, glob, io, os, zipfile
import torch
from google.colab import drive

assert torch.cuda.is_available(), "No hay GPU: elige GPU T4 en Entorno de ejecución y vuelve a ejecutar."
print("GPU:", torch.cuda.get_device_name(0))

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/tabd_scale_e5_base"
os.makedirs(OUT, exist_ok=True)

BUNDLE = """
...
ALoBAAD1fAMAAAA=
"""
os.makedirs("/content/tabd", exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))).extractall("/content/tabd")
os.chdir("/content/tabd")
print("Archivos listos en", os.getcwd())

In [ ]:
#@title 2. Instalar dependencias
!pip install -q datasets sentence-transformers pandas pyarrow

In [ ]:
#@title 3. Elegir qué tramos generar
PRIMER_TRAMO = 7  #@param {type:"integer"}
ULTIMO_TRAMO = 10  #@param {type:"integer"}
CHUNK_VECTORS = 50_000

wanted = range(PRIMER_TRAMO, ULTIMO_TRAMO + 1)
missing = sum(not os.path.exists(f"{OUT}/part_e5_base_{(t - 1) * CHUNK_VECTORS:08d}.npy") for t in wanted)
START_AT = (PRIMER_TRAMO - 1) * CHUNK_VECTORS
STOP_AFTER = ULTIMO_TRAMO * CHUNK_VECTORS
print(f"Tramos {PRIMER_TRAMO} a {ULTIMO_TRAMO}: faltan {missing}.")
print(f"Comprueba que Drive tenga libres al menos {missing * 150 + 300} MB: Colab no puede medirlo aquí.")

In [ ]:
#@title 4. Vectorizar (varias horas; se puede retomar)
!python colab_embed_scale.py --model e5_base --n 1000000 --start-at {START_AT} --stop-after {STOP_AFTER} --batch-size 64 --out "{OUT}"

In [ ]:
#@title 5. Terminar de subir a Drive
drive.flush_and_unmount()
print("Drive sincronizado. Revisa la carpeta tabd_scale_e5_base y avísame cuántos tramos quedaron.")